# 8 · Inequality tasks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rudip1/learn-manipulator-control/blob/main/2_notebooks/exercises/08_inequality_tasks.ipynb)

Theory: [`1_theory/08_inequality_tasks.md`](../../1_theory/08_inequality_tasks.md) · notation:
[`00_notation.md`](../../1_theory/00_notation.md)

In [ ]:
# Install the module when it is not importable yet (Colab, fresh environment).
import importlib.util
if importlib.util.find_spec("manipulator_control") is None:
    %pip install -q git+https://github.com/Rudip1/learn-manipulator-control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import manipulator_control as mc
from manipulator_control import plotting

plt.rcParams.update({"figure.dpi": 80, "axes.grid": True})
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(8)

## Recap

- An inequality task keeps $\sigma_{\min} \le \sigma(q) \le \sigma_{\max}$ (eq. 8.1). It joins the hierarchy only
  while it is near a bound (the active set, eq. 8.2), at the top.
- Activation (eq. 8.3): on within $\alpha$ of a bound, off once $\delta$ away; $\delta > \alpha$ is hysteresis.
  While active it asks for $v = Ka$, $a = \pm1$: a constant push away from the bound (eq. 8.4).
- Joint limit: $J = e_j^T$ (8.5). Obstacle: $\sigma = \|p - c\|$, $J = (p - c)^TJ_P/\|p - c\|$ (8.6).
- No hysteresis → chattering; no margin → the bound is crossed by one step.

## The activation as a hysteresis loop

Sweep $\sigma$ up to the upper bound and back down, and record the activation state (`mc.next_activation`).

In [ ]:
sweep = np.concatenate([np.linspace(0.5, 0.98, 200), np.linspace(0.98, 0.5, 200)])
fig, ax = plt.subplots(figsize=(6, 3))
for alpha, delta, style in [(0.1, 0.3, "-"), (0.1, 0.1, "--")]:
    s, states = 0, []
    for x in sweep:
        s = mc.next_activation(s, x, -1.0, 1.0, alpha, delta)
        states.append(s)
    ax.plot(sweep[:200], np.abs(states[:200]), style, color="C0", label=f"up, α={alpha}, δ={delta}")
    ax.plot(sweep[200:], np.abs(states[200:]) - 0.03, style, color="C1", label=f"down, α={alpha}, δ={delta}")
ax.set_xlabel("σ (upper bound 1.0)"); ax.set_ylabel("|activation|"); ax.legend(fontsize=7); plt.show()

## Joint limit above the tip task

The tip of the three-link arm moves to $(0.6, 0.5)$ m; without a limit joint 3 reaches 1.859 rad. Limit it to
$\pm1.7$ rad.

In [ ]:
arm = mc.robots.planar([0.5, 0.4, 0.3])
model = mc.SerialArmModel(arm)
q0 = np.array([1.2, -1.5, 1.6])
target = np.eye(4); target[:2, 3] = [0.6, 0.5]

def limit_run(alpha=0.05, delta=0.15, with_limit=True, limit_first=True, steps=1000):
    tip = mc.FrameTask("tip", -1, [0, 1], target); tip.set_gain(2.0)
    limit = mc.JointLimitTask("q3 limit", 2, -1.7, 1.7, alpha, delta); limit.set_gain(0.5)
    tasks = ([limit, tip] if limit_first else [tip, limit]) if with_limit else [tip]
    return mc.simulate_task_priority(model, q0, tasks, mc.PriorityConfig(), 0.01, steps)

switches = lambda active: int(np.abs(np.diff(active)).sum())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, r in [("no limit", limit_run(with_limit=False)), ("α=0.05, δ=0.15", limit_run())]:
    ax[0].plot(r.t, r.q[:, 2], label=name)
    ax[1].plot(r.t[:-1], np.abs(r.u[:-1]).max(axis=1), label=name)
    print(f"{name:15s} max q3 = {r.q[:, 2].max():.4f} rad, tip error = {r.error_norm[-1, -1]:.1e} m")
ax[0].axhline(1.7, color="k"); ax[0].set_title("q3 [rad]"); ax[1].set_title("max |qdot| [rad/s]")
ax[0].legend(); plt.show()

## Obstacle avoidance

An obstacle of radius 0.12 m sits on the path of the joint-3 point. The obstacle task watches that point;
the tip task is below it.

In [ ]:
q0_obs = np.array([-0.4, 1.2, 0.8])
target_obs = np.eye(4); target_obs[:2, 3] = [0.2, 0.75]
c, radius = np.array([0.5205, 0.4496, 0.0]), 0.12
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for k, with_obstacle in enumerate([False, True]):
    tip = mc.FrameTask("tip", -1, [0, 1], target_obs); tip.set_gain(2.0)
    obstacle = mc.ObstacleTask("obstacle", 2, c, radius, 0.03, 0.08, True); obstacle.set_gain(0.5)
    r = mc.simulate_task_priority(model, q0_obs, [obstacle, tip] if with_obstacle else [tip], mc.PriorityConfig(), 0.01, 800)
    p3 = np.array([arm.frames(q)[2][:2, 3] for q in r.q])
    ax[0].plot(*p3.T, color=f"C{k}"); plotting.draw_planar_arm(ax[0], arm.frames(r.q[-1]), color=f"C{k}")
    ax[1].plot(r.t, np.linalg.norm(p3 - c[:2], axis=1), color=f"C{k}", label=f"obstacle task: {with_obstacle}")
ax[0].add_patch(plt.Circle(c[:2], radius, color="0.3", alpha=0.5)); ax[0].plot(*target_obs[:2, 3], "r*", markersize=12)
plotting.planar_axes(ax[0], limit=1.0); ax[1].axhline(radius, color="k"); ax[1].legend(); ax[1].set_title("distance [m]")
plt.show()

## ✏️ Exercises

### ✏️ 1. The activation state machine

Implement eq. (8.3).

In [ ]:
def activation(state, sigma, s_min, s_max, alpha, delta):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
for _ in range(200):
    s_ref = s_mine = 0
    for x in rng.uniform(-1.2, 1.2, 50):
        s_ref = mc.next_activation(s_ref, x, -1.0, 1.0, 0.1, 0.25)
        s_mine = activation(s_mine, x, -1.0, 1.0, 0.1, 0.25)
        assert s_ref == s_mine
print("✓ activation")

### ✏️ 2. The distance Jacobian

Implement eq. (8.6) for the planar arm: distance of DH frame `k`'s origin from `c` in the $(x, y)$ plane and its
$1\times n$ Jacobian. Use `mc.link_jacobian(arm, q, k)` for $J_P$.

In [ ]:
def distance_task(q, k, c):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
for _ in range(20):
    q = rng.uniform(-np.pi, np.pi, 3)
    task = mc.ObstacleTask("obs", 2, c, radius, 0.03, 0.08, True)
    task.update(model, q)
    sigma, J = distance_task(q, 2, c)
    assert np.isclose(sigma, task.value) and np.allclose(J, task.jacobian[0])
print("✓ distance_task")

### ✏️ 3. How large must $\alpha$ be?

The task reacts one control period after $\sigma$ enters the band. In the joint-limit example joint 3 moves at
most `qdot_max` (measure it on the run without limit task); with $\Delta t = 10$ ms, what margin $\alpha$
guarantees that the limit is never crossed? Check it by simulation.

In [ ]:
free = limit_run(with_limit=False)
qdot_max = np.abs(free.u[:, 2]).max()
# ✏️ your code here
raise NotImplementedError("exercise")

In [ ]:
r = limit_run(alpha=alpha_min, delta=alpha_min + 0.1)
print(f"qdot_max = {qdot_max:.3f} rad/s -> alpha_min = {alpha_min * 1e3:.1f} mrad; max q3 = {r.q[:, 2].max():.5f}")
assert np.isclose(alpha_min, 0.01 * qdot_max) and r.q[:, 2].max() <= 1.7
print("✓ alpha_min")

## 🔨 Break it

### 🔨 1. No hysteresis

Set $\delta = \alpha$. The limit still holds, but the task switches dozens of times and the command jumps at every
switch.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for delta in [0.15, 0.05]:
    r = limit_run(alpha=0.05, delta=delta, steps=300)
    ax[0].step(r.t, r.active[:, 0], where="post", label=f"δ = {delta}: {switches(r.active[:, 0])} switches")
    ax[1].plot(r.t[:-1], r.u[:-1, 2], label=f"δ = {delta}")
ax[0].set_title("limit task active"); ax[1].set_title("joint-3 rate [rad/s]"); ax[0].legend(); plt.show()

### 🔨 2. No margin

$\alpha = \delta = 0$: the task acts only after the limit is crossed.

In [ ]:
r = limit_run(alpha=0.0, delta=0.0)
print(f"max q3 = {r.q[:, 2].max():.4f} rad (limit 1.7)")

### 🔨 3. Safety below the end-effector task

Put an obstacle (radius 0.1 m) over the tip's own target, so the two tasks really conflict. With the obstacle task
on top the tip stops at the obstacle; with the tip task on top the tip goes in. (DLS is used for both orders: a
conflicting task *below* is an algorithmic singularity of chapter 7, and with the pseudoinverse the joints run
away.) Note also the switch count with the obstacle on top: the tip task keeps pushing into the bound, so the
obstacle task cycles on and off — hysteresis sets the period, it cannot remove the cycle.

In [ ]:
q0_c = np.array([0.3, 0.7, 0.6])
c_t = np.array([0.65, 0.5, 0.0])
dls = mc.PriorityConfig(inverse=mc.InverseMethod.DampedLeastSquares, damping=0.05)
for obstacle_first in [True, False]:
    tip = mc.FrameTask("tip", -1, [0, 1], target); tip.set_gain(2.0)
    obstacle = mc.ObstacleTask("obstacle", -1, c_t, 0.1, 0.02, 0.05, True); obstacle.set_gain(0.5)
    tasks = [obstacle, tip] if obstacle_first else [tip, obstacle]
    r = mc.simulate_task_priority(model, q0_c, tasks, dls, 0.01, 800)
    d = np.linalg.norm(r.ee_position[:, :2] - c_t[:2], axis=1)
    k = 0 if obstacle_first else 1
    print(f"obstacle first = {obstacle_first!s:5s}: closest approach {d.min():.3f} m (radius 0.1), {switches(r.active[:, k])} switches")

### 🔨 4. A proportional pull that never lets go

Replace the constant push (8.4) by a pull to the release threshold, $v = K(\sigma_{\max} - \delta - \sigma)$, built
here from the library's solver and activation function. Give the tip two goals in a row. The pull approaches the
threshold exponentially, never crosses it, and the task stays active for good: joint 3 is frozen at 1.55 rad,
while with the constant push it is released and moves freely for the second goal.

In [ ]:
T1, T2 = np.eye(4), np.eye(4)
T1[:2, 3], T2[:2, 3] = [0.6, 0.5], [0.9, 0.3]

def run_proportional(goals, steps_each=600, dt=0.01, alpha=0.05, delta=0.15):
    q, state, log = q0.copy(), 0, []
    for T in goals:
        for _ in range(steps_each):
            state = mc.next_activation(state, q[2], -1.7, 1.7, alpha, delta)
            levels = [mc.PriorityLevel(mc.geometric_jacobian(arm, q)[:2], 2.0 * (T[:2, 3] - arm.end_effector(q)[:2, 3]))]
            if state != 0:   # pull to the release threshold instead of pushing past it
                levels.insert(0, mc.PriorityLevel(np.array([[0.0, 0.0, 1.0]]), np.array([0.5 * ((1.7 - delta) - q[2])])))
            q = q + dt * mc.task_priority_velocity(levels, 3)
            log.append((q[2], state != 0))
    return np.array(log)

def run_push(goals, steps_each=600):
    limit = mc.JointLimitTask("q3", 2, -1.7, 1.7, 0.05, 0.15); limit.set_gain(0.5)
    tip = mc.FrameTask("tip", -1, [0, 1], goals[0]); tip.set_gain(2.0)
    q, q3, active = q0.copy(), [], []
    for T in goals:
        tip.desired = T
        r = mc.simulate_task_priority(model, q, [limit, tip], mc.PriorityConfig(), 0.01, steps_each)
        q = r.q[-1]; q3.append(r.q[1:, 2]); active.append(r.active[1:, 0])
    return np.concatenate(q3), np.concatenate(active)

prop = run_proportional([T1, T2])
push_q3, push_active = run_push([T1, T2])
t = np.arange(len(prop)) * 0.01
plt.figure(figsize=(7, 3.2))
plt.plot(t, prop[:, 0], label="proportional pull"); plt.plot(t, push_q3, label="constant push (8.4)")
plt.axhline(1.55, color="0.6", linestyle="--"); plt.axvline(6.0, color="0.8")
plt.xlabel("t [s] (second goal from 6 s)"); plt.ylabel("q3 [rad]"); plt.legend(fontsize=8); plt.show()
print("still active at the end — proportional:", bool(prop[-1, 1]), "  constant push:", bool(push_active[-1]))

## What to remember

- An inequality task joins the hierarchy only near a bound, and then at the top: it costs redundancy only when
  needed.
- $\alpha > 0$ covers the one-step delay; $\delta > \alpha$ (hysteresis) prevents chattering.
- While active, push at a constant rate until the release threshold is crossed — a proportional pull never lets
  go.
- Set-based switching still makes the command jump; continuous activations and hierarchical QPs are the
  smoother (and costlier) alternatives.